# MLP Test

## Notes
- Loads `mlp_best.pt` to apply to test.

In [1]:
# General Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from math import ceil
from time import perf_counter
from datetime import datetime

import json

In [2]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report, f1_score

- ✅ MLP Class definition
- ✅ Fit the `StandardScaler` on `df_train[feature_cols]`
- ✅ `THRESHOLD = 0.5` inclusion.
- ✅ `evaluate()`

In [3]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.flatten = nn.Flatten()       # Flattens the input tensor to a 1D tensor
        self.hidden  = nn.Linear(252, 32) # Mapping 252 input features (x_n) to 32 hidden neurons
        self.output  = nn.Linear(32, 2)   # Mapping 32 hidden neurons to 2 output classes (normal/abnormal)
    
    def forward(self, x):
        z = self.hidden(self.flatten(x))
        h = torch.relu(z)
        return self.output(h)

In [ ]:
# Load ECG
TRAIN_ECG  = np.load('train_ecg.npz')
TRAIN_ECGx = TRAIN_ECG['X']
TRAIN_ECGy = TRAIN_ECG['y']

# Test ECG
assert Path('test_ecg.npz').is_file(), "test_ecg.npz was not found"
TEST_ECG  = np.load('test_ecg.npz')
TEST_ECGx = TEST_ECG['X']
TEST_ECGy = TEST_ECG['y']

# DataFrame creation from 
df_train = pd.DataFrame(TRAIN_ECGx, columns=[f'x_{i+1}' for i in range(TRAIN_ECGx.shape[1])])
df_train['target'] = TRAIN_ECGy

df_test = pd.DataFrame(TEST_ECGx, columns=[f'x_{i+1}' for i in range(TEST_ECGx.shape[1])])
df_test['target'] = TEST_ECGy

In [ ]:
# Scaler
feature_cols = [c for c in df_train.columns if c != 'target']
scaler = StandardScaler().fit(df_train[feature_cols])

def to_tensor_dataset(df):
    '''
    Substitutes **W11_D2**'s `balanced_subset()` function. \n
    Converts a DataFrame into a PyTorch TensorDataset.
    '''
    # Extract features and labels
    X = torch.tensor(scaler.transform(df[feature_cols]), dtype=torch.float32)
    y = torch.tensor(df['target'].values, dtype=torch.long)
    
    return TensorDataset(X, y)

In [ ]:
train_data = to_tensor_dataset(df_train)
test_data  = to_tensor_dataset(df_test)

train_loader = DataLoader(train_data, batch_size=512, shuffle=False)
test_loader = DataLoader(test_data, batch_size=512, shuffle=False)

In [ ]:
print(f"Train DataLoader: {len(train_loader)} batches, each with {train_loader.batch_size} samples")
print(f"Test DataLoader: {len(test_loader)} batches, each with {test_loader.batch_size} samples")

In [5]:
SEED = 825
torch.manual_seed(SEED)

torch.cuda.manual_seed_all(SEED) # Set the random seed for reproducibility
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Set the device to GPU if available, otherwise CPU

# Class wights for training set frequencies
class_counts = np.bincount(TRAIN_ECGy)
class_weights = len(TRAIN_ECGy) / (len(class_counts) * class_counts)
class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
    ).to(device)
print(f"Class weights: \n{class_weights}")

criterion      = nn.CrossEntropyLoss(weight=class_weights) # Loss Function for multi-class classification
eval_criterion = nn.CrossEntropyLoss()

Class weights: 
tensor([0.5516, 5.3482], device='cuda:0')


In [6]:
THRESHOLD = 0.5

In [7]:
def evaluate(model, loader):
    model.eval()
    total, probs, labels = 0.0, [], []
    with torch.inference_mode():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            total += eval_criterion(logits, y_batch).item() * len(y_batch) # criterion(logits, y_batch).item() * len(y_batch)
            probs.append(torch.softmax(logits, dim=1)[:, 1].cpu())  # P(abnormal)
            labels.append(y_batch.cpu())
    return total / len(loader.dataset), torch.cat(probs).numpy(), torch.cat(labels).numpy()

### Load Best Model

In [8]:
MODEL = MLP()

state_dict = torch.load('mlp_best.pt', map_location=device, weights_only=True)

MODEL.load_state_dict(state_dict)
MODEL.to(device)

MLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (hidden): Linear(in_features=252, out_features=32, bias=True)
  (output): Linear(in_features=32, out_features=2, bias=True)
)

In [ ]:
test_loss, test_probs, test_labels = evaluate(MODEL, test_loader)
test_preds = (test_probs >= THRESHOLD).astype(int)
print(f"Test loss {test_loss:.4f} | ROC AUC {roc_auc_score(test_labels, test_probs):.3f}")
print(classification_report(test_labels, test_preds, digits=3))
print(confusion_matrix(test_labels, test_preds))